# Download Landsat / Sentinel-2 indices at 2 m in S-JTSK Krovak

# This script downloads Landsat (2001, 2009) and Sentinel-2 (2020) satellite imagery for a study area, computes four spectral indices (NDVI, NDWI, NDBI, NDPI) from clear-sky scenes, and resamples them to a uniform 2 m resolution grid in the S-JTSK / Krovak East North coordinate system (EPSG:5514). The final GeoTIFF outputs are aligned to a common spatial grid so they can be used directly with other local datasets such as land-use/land-cover maps.
```


## 1. Imports


In [ ]:
from pathlib import Path
import warnings
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.crs import CRS
from rasterio.transform import from_origin
from rasterio.warp import reproject, Resampling
from shapely.ops import unary_union

import planetary_computer as pc
from pystac_client import Client
import stackstac

warnings.filterwarnings("ignore")
print("Libraries loaded")


## 2. Paths and CRS (edit paths)


In [ ]:
# ========== EDIT PATHS ==========
# Basin boundary shapefile (any CRS is fine; will be reprojected)
BASIN = r"./data/Gidra_basin_boundary.shp"

# Optional: if this LULC is already EPSG:5514 and 2 m, its grid is reused
TARGET_GRID = r"./data/LULC_2020.tif"

OUT_DIR = Path(r"./output/spectral_indices")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Same as LULC2009.prj -> S-JTSK Krovak East North
OUTPUT_CRS = CRS.from_epsg(5514)
CELL_SIZE_M = 2.0

PERIODS = {
    2001: {"start": "2001-05-01", "end": "2001-09-30", "sensor": "landsat", "max_cloud": 20},
    2009: {"start": "2009-05-01", "end": "2009-09-30", "sensor": "landsat", "max_cloud": 20},
    2020: {"start": "2020-05-01", "end": "2020-09-30", "sensor": "sentinel2", "max_cloud": 15},
}
LANDSAT_RES = 30
S2_RES = 10
MAX_ITEMS = 20

print("OUTPUT_CRS:", OUTPUT_CRS)
print("CELL_SIZE_M:", CELL_SIZE_M)
print("OUT_DIR:", OUT_DIR)


## 3. Study area + 2 m grid in EPSG:5514


In [ ]:
basin = gpd.read_file(BASIN)
if basin.crs is None:
    raise ValueError("Basin shapefile has no CRS")

basin_wgs = basin.to_crs(4326)
geom = unary_union(basin_wgs.geometry)
minx, miny, maxx, maxy = geom.bounds
pad = 0.01
bbox = [minx - pad, miny - pad, maxx + pad, maxy + pad]
print("WGS84 bbox (STAC search):", bbox)

use_ref = False
if Path(TARGET_GRID).exists():
    with rasterio.open(TARGET_GRID) as ref:
        ref_crs = ref.crs
        ref_res = abs(ref.transform.a)
        print(f"TARGET_GRID CRS={ref_crs}, res={ref_res}")
        same = False
        try:
            if ref_crs and ref_crs.to_epsg() == 5514:
                same = True
        except Exception:
            pass
        if ref_crs and ("S-JTSK" in str(ref_crs).upper() or "KROVAK" in str(ref_crs).upper()):
            same = True
        if same and abs(ref_res - CELL_SIZE_M) < 0.15:
            dst_crs = ref_crs
            dst_transform = ref.transform
            dst_shape = (ref.height, ref.width)
            use_ref = True
            print("Using TARGET_GRID extent/transform (2 m, S-JTSK).")
        else:
            print("TARGET_GRID differs — building new 2 m EPSG:5514 grid from basin.")

if not use_ref:
    basin_sjtsk = basin.to_crs(OUTPUT_CRS)
    minx, miny, maxx, maxy = basin_sjtsk.total_bounds
    pad_m = 200.0
    minx -= pad_m
    miny -= pad_m
    maxx += pad_m
    maxy += pad_m
    width = int(np.ceil((maxx - minx) / CELL_SIZE_M))
    height = int(np.ceil((maxy - miny) / CELL_SIZE_M))
    dst_transform = from_origin(minx, maxy, CELL_SIZE_M, CELL_SIZE_M)
    dst_shape = (height, width)
    dst_crs = OUTPUT_CRS
    print(f"New grid: shape={dst_shape}")

print("Output CRS:", dst_crs)
print("Output shape:", dst_shape)
print("Output res: 2 m x 2 m")


## 4. STAC client


In [ ]:
catalog = Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=pc.sign_inplace,
)
print("STAC OK")


## 5. Search + index helpers


In [ ]:
def search_items(sensor, start, end, bbox, max_cloud, max_items=MAX_ITEMS):
    coll = "landsat-c2-l2" if sensor == "landsat" else "sentinel-2-l2a"
    search = catalog.search(
        collections=[coll],
        bbox=bbox,
        datetime=f"{start}/{end}",
        query={"eo:cloud_cover": {"lt": max_cloud}},
        max_items=max_items * 3,
    )
    items = list(search.items())
    items = sorted(items, key=lambda it: it.properties.get("eo:cloud_cover", 100))
    return items[:max_items]


def pick_assets(item, sensor):
    keys = item.assets.keys()
    if sensor == "landsat":
        cand = {
            "green": ["green", "SR_B3", "B3"],
            "red": ["red", "SR_B4", "B4"],
            "nir": ["nir08", "nir", "SR_B5", "B5"],
            "swir": ["swir16", "SR_B6", "B6"],
            "qa": ["qa_pixel", "QA_PIXEL"],
        }
    else:
        cand = {
            "green": ["B03", "B3"],
            "red": ["B04", "B4"],
            "nir": ["B08", "B8"],
            "swir": ["B11"],
            "qa": ["SCL"],
        }
    out = {}
    for k, opts in cand.items():
        for o in opts:
            if o in keys:
                out[k] = o
                break
    return out


def build_indices(items, sensor, resolution, bbox_wgs):
    names = pick_assets(items[0], sensor)
    need = ["green", "red", "nir", "swir"]
    if any(k not in names for k in need):
        raise RuntimeError(f"Band map incomplete: {names}. Assets: {list(items[0].assets)}")
    print("  bands:", names)

    asset_list = [names[k] for k in need]
    if "qa" in names:
        asset_list = asset_list + [names["qa"]]

    utm = basin.estimate_utm_crs()
    epsg = int(str(utm).split(":")[-1])

    stack = stackstac.stack(
        items,
        assets=asset_list,
        resolution=resolution,
        epsg=epsg,
        bounds_latlon=tuple(bbox_wgs),
        chunksize=2048,
        rescale=False,
    )

    optical = stack.sel(band=[names[k] for k in need]).astype("float32")

    if sensor == "landsat":
        optical = (optical * 0.0000275 - 0.2).clip(0, 1)
        if "qa" in names:
            qa = stack.sel(band=names["qa"]).astype("int32")
            clear = (
                ((qa & (1 << 1)) == 0)
                & ((qa & (1 << 2)) == 0)
                & ((qa & (1 << 3)) == 0)
                & ((qa & (1 << 4)) == 0)
            )
            optical = optical.where(clear)
    else:
        if float(optical.max().values) > 2:
            optical = optical / 10000.0
        optical = optical.clip(0, 1)
        if "qa" in names:
            scl = stack.sel(band=names["qa"])
            optical = optical.where(scl.isin([4, 5, 6, 7]))

    med = optical.median(dim="time", skipna=True)
    green = med.sel(band=names["green"])
    red = med.sel(band=names["red"])
    nir = med.sel(band=names["nir"])
    swir = med.sel(band=names["swir"])
    eps = 1e-6
    indices = {
        "NDVI": ((nir - red) / (nir + red + eps)).compute(),
        "NDWI": ((green - nir) / (green + nir + eps)).compute(),
        "NDBI": ((swir - nir) / (swir + nir + eps)).compute(),
        "NDPI": ((nir - swir) / (nir + swir + eps)).compute(),
    }
    return indices, utm


def save_native_utm(da, path, crs):
    arr = np.asarray(da.values, dtype="float32")
    y = np.asarray(da.y.values)
    x = np.asarray(da.x.values)
    res_x = abs(float(x[1] - x[0])) if len(x) > 1 else 30.0
    res_y = abs(float(y[0] - y[1])) if len(y) > 1 else 30.0
    transform = from_origin(float(x.min()), float(y.max()), res_x, res_y)
    profile = {
        "driver": "GTiff",
        "height": arr.shape[0],
        "width": arr.shape[1],
        "count": 1,
        "dtype": "float32",
        "crs": crs,
        "transform": transform,
        "compress": "deflate",
        "nodata": np.nan,
    }
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(arr, 1)
    return path


def warp_to_sjtsk_2m(src_path, dst_path):
    dst = np.full(dst_shape, np.nan, dtype="float32")
    with rasterio.open(src_path) as src:
        reproject(
            source=src.read(1).astype("float32"),
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=dst_transform,
            dst_crs=dst_crs,
            resampling=Resampling.bilinear,
            src_nodata=src.nodata,
            dst_nodata=np.nan,
        )
    profile = {
        "driver": "GTiff",
        "height": dst_shape[0],
        "width": dst_shape[1],
        "count": 1,
        "dtype": "float32",
        "crs": dst_crs,
        "transform": dst_transform,
        "compress": "deflate",
        "nodata": np.nan,
    }
    with rasterio.open(dst_path, "w", **profile) as out:
        out.write(dst, 1)
    print(f"  -> {Path(dst_path).name}  CRS=EPSG:5514  res=2m  shape={dst_shape}")


## 6. Download and write EPSG:5514 / 2 m GeoTIFFs


In [ ]:
results = {}
for year, cfg in PERIODS.items():
    print(f"\n======== {year} ({cfg['sensor']}) ========")
    items = search_items(cfg["sensor"], cfg["start"], cfg["end"], bbox, cfg["max_cloud"])
    print(f"Scenes: {len(items)}")
    if not items:
        print("SKIP — raise max_cloud or widen dates")
        continue
    res = LANDSAT_RES if cfg["sensor"] == "landsat" else S2_RES
    try:
        indices, utm_crs = build_indices(items, cfg["sensor"], res, bbox)
    except Exception as e:
        print("FAILED:", type(e).__name__, e)
        continue

    results[year] = {}
    for name, da in indices.items():
        native = OUT_DIR / f"{name}_{year}_native.tif"
        out_2m = OUT_DIR / f"{name}_{year}_2m.tif"
        save_native_utm(da, native, utm_crs)
        warp_to_sjtsk_2m(str(native), str(out_2m))
        results[year][name] = str(out_2m)

print("\n======== DONE ========")
for y, d in results.items():
    print(y, list(d.keys()))

sample = OUT_DIR / "NDVI_2020_2m.tif"
if sample.exists():
    with rasterio.open(sample) as s:
        print(f"\nVerify {sample.name}:")
        print(f"  shape={s.shape}")
        print(f"  res={s.res}")
        print(f"  CRS={s.crs}")
        try:
            print(f"  EPSG={s.crs.to_epsg()}")
        except Exception:
            pass


## 7. Outputs

All `*_2m.tif` files:

- **CRS:** S-JTSK / Krovak East North (**EPSG:5514**) — same as `LULC2009.prj`
- **Cell size:** 2 x 2 m
- **Indices:** NDVI, NDWI, NDBI, NDPI for 2001, 2009, 2020

No TWI / DEM in this script.
